In [0]:
# # --- Configuration ---
# BU_Group="COMMERCIALS"
# bu_filter = ['COMCE-GERMANY,AUSTRIA, SWITZERL,CEE', 'COMAS-ASIA', 'GLOBA-GLOBAL','COMNA-NORTH AMERICA','COMNN-NETHERLANDS','COMON-OCEANIA','COMSE-BELGIUM, LUXEMBOURG','COMSPB-SPAIN, PORTUGAL, BRAZIL','COMUI-UK & IRELAND','FRANC-FRANCE','ITALY-ITALY','CREDIT INSURANCE' ]

# # # --- Configuration ---
# # BU_Group="RISK"
# # bu_filter = ['RISK1-RS1-GERMANY, CENTRAL & EAST EUROPE','RISK2-RS2-BELGIUM, LUX, FRANCE & ITA','RISK3-RS3-NLD, APAC & AIM','RISK4-RS4-NORTH EUROPE, CIS & ACS','RISK4-RS4-NORTH EUROPE, CIS & SP','RISK5-RS5-NAFTA','RISK1-RS1-DEU, AUT and CHE','RISK1-RS1-Europe, Russia & CIS','RISK2-RS2-NLD, Belux, FRA, Africa & ISR','RISK3-RS3-Asia and Oceania','RISK7-RS7-Spain, Portugal, Andorra']

# --- Configuration ---
BU_Group="FINANCE"
bu_filter = ['FINPM-FINANCE PROGRAM MANAGEMENT','GFO-GROUP FINANCE OPERATIONS','GFC-GROUP FINANCE AND CONTROL','COFIN-CORPORATE FINANCE & TAX','Group Finance','Finance','Finance & Control']

# SQL-friendly string for Python f-string IN clauses: 'val1','val2'
bu_filter_sql = "','".join([b.upper() for b in bu_filter])

# Create temp view for SQL cells to reference (avoids widget quote escaping)
from pyspark.sql import Row
bu_df = spark.createDataFrame([Row(bu=b) for b in bu_filter])
bu_df.createOrReplaceTempView("vw_bu_filter")

print(f"BU filter values: {bu_filter}")
print(f"SQL string: IN ('{bu_filter_sql}')")
print("SQL cells: use WHERE UP.BU IN (SELECT bu FROM vw_bu_filter)")

In [0]:
%sql
select distinct BU from dataplatform01_central_dev_catalog_01.custom_sap_bo.ad_user_profiles 
order by BU asc;

  SELECT DISTINCT UP.BU
  FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata AS CMS
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit AS UA
    ON UPPER(TRIM(CMS.Document_Id)) = UPPER(TRIM(CAST(UA.WEBI_Doc_ID AS STRING)))
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS UP
    ON UPPER(TRIM(UP.UserName)) = UPPER(TRIM(UA.UserName))
  WHERE UPPER(TRIM(cms.Document_name)) NOT IN ('DOCUMENT NOT FOUND ON SERVER')
    AND UPPER(UP.BU) IN (SELECT UPPER(bu) FROM vw_bu_filter)
ORDER BY BU

In [0]:
%sql
SELECT DISTINCT
UP.BU,
cms.Document_Id,
CASE WHEN S1.document_id IS NULL THEN 'N' ELSE 'Y' END AS scheduled
FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata AS CMS
LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS UP
ON UPPER(TRIM(
        CASE WHEN UPPER(TRIM(CMS.created)) = 'ADMINISTRATOR'
            THEN CMS.lastAuthor
            ELSE CMS.created
        END
    )) = UPPER(TRIM(UP.UserName))
LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_extracted AS S1
ON UPPER(TRIM(CMS.Document_Id)) = UPPER(TRIM(S1.document_id))
WHERE UPPER(TRIM(cms.Document_name)) NOT IN ('DOCUMENT NOT FOUND ON SERVER')
AND UPPER(UP.BU) IN (SELECT UPPER(bu) FROM vw_bu_filter)
ORDER BY up.BU asc

-- select BU, count(distinct Document_Id) as cnt from _sqldf
-- group by BU

In [0]:
%sql
WITH Owned_Reports AS (
  SELECT DISTINCT
    cms.Document_Id,
    CASE WHEN S1.document_id IS NULL THEN 'N' ELSE 'Y' END AS scheduled
  FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata AS CMS
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS UP
    ON UPPER(TRIM(
          CASE WHEN UPPER(TRIM(CMS.created)) = 'ADMINISTRATOR'
               THEN CMS.lastAuthor
               ELSE CMS.created
          END
      )) = UPPER(TRIM(UP.UserName))
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_extracted AS S1
    ON UPPER(TRIM(CMS.Document_Id)) = UPPER(TRIM(S1.document_id))
  WHERE UPPER(TRIM(cms.Document_name)) NOT IN ('DOCUMENT NOT FOUND ON SERVER')
    AND UPPER(UP.BU) IN (SELECT UPPER(bu) FROM vw_bu_filter)
)

-- Access-only documents (not owned by BU users)
SELECT
  'Only Access' AS category,
  COUNT(DISTINCT CMS.Document_Id) AS Document_cnt
FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata AS CMS
LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit AS UA
  ON UPPER(TRIM(CMS.Document_Id)) = UPPER(TRIM(CAST(UA.WEBI_Doc_ID AS STRING)))
LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS UP
  ON UPPER(TRIM(UP.UserName)) = UPPER(TRIM(UA.UserName))
WHERE UPPER(TRIM(cms.Document_name)) NOT IN ('DOCUMENT NOT FOUND ON SERVER')
  AND UPPER(UP.BU) IN (SELECT UPPER(bu) FROM vw_bu_filter)
  AND cms.Document_Id NOT IN (SELECT DISTINCT Document_Id FROM Owned_Reports)

UNION ALL
-- Access-own documents
SELECT
  'Access Owned' AS category,
  COUNT(DISTINCT CMS.Document_Id) AS Document_cnt
FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata AS CMS
LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit AS UA
  ON UPPER(TRIM(CMS.Document_Id)) = UPPER(TRIM(CAST(UA.WEBI_Doc_ID AS STRING)))
LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS UP
  ON UPPER(TRIM(UP.UserName)) = UPPER(TRIM(UA.UserName))
WHERE UPPER(TRIM(cms.Document_name)) NOT IN ('DOCUMENT NOT FOUND ON SERVER')
  AND UPPER(UP.BU) IN (SELECT UPPER(bu) FROM vw_bu_filter)
  AND cms.Document_Id IN (SELECT DISTINCT Document_Id FROM Owned_Reports)
;

  WITH Owned_Reports AS (
    SELECT DISTINCT
      cms.Document_Id,
      CASE WHEN S1.document_id IS NULL THEN 'N' ELSE 'Y' END AS scheduled,
      S1.Active_Schedule as schedule_active
    FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata AS CMS
    LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS UP
      ON UPPER(TRIM(
          CASE WHEN UPPER(TRIM(CMS.created)) = 'ADMINISTRATOR'
               THEN CMS.lastAuthor
               ELSE CMS.created
          END
      )) = UPPER(TRIM(UP.UserName))
    LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_extracted AS S1
      ON UPPER(TRIM(CMS.Document_Id)) = UPPER(TRIM(S1.document_id))
    WHERE UPPER(TRIM(cms.Document_name)) NOT IN ('DOCUMENT NOT FOUND ON SERVER')
      AND UPPER(UP.BU) IN (SELECT UPPER(bu) FROM vw_bu_filter) 
  )

  SELECT
    'Total owned normal reports' AS category,
    COUNT(DISTINCT Document_Id) AS Document_cnt
  FROM Owned_Reports
  WHERE scheduled = 'N'
  or( scheduled ='Y' and schedule_active = false)

  UNION ALL

  SELECT
    'Total owned scheduled reports' AS category,
    COUNT(DISTINCT Document_Id) AS Document_cnt
  FROM Owned_Reports
  WHERE scheduled = 'Y'
  AND  schedule_active = true;

SELECT
  COUNT(DISTINCT UP.UserName) AS User_Cnt
FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata AS CMS
LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit AS UA
  ON UPPER(TRIM(CMS.Document_Id)) = UPPER(TRIM(CAST(UA.WEBI_Doc_ID AS STRING)))
LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS UP
  ON UPPER(TRIM(UP.UserName)) = UPPER(TRIM(UA.UserName))
WHERE UPPER(TRIM(cms.Document_name)) NOT IN ('DOCUMENT NOT FOUND ON SERVER')
  AND UPPER(UP.BU) IN (SELECT UPPER(bu) FROM vw_bu_filter)

In [0]:
%sql
    
    
WITH owned AS (
    SELECT DISTINCT co.cluster, CAST(cms.Document_Id AS STRING) AS Doc_ID
    FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata AS CMS
    LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS UP
      ON UPPER(TRIM(
          CASE WHEN UPPER(TRIM(CMS.created)) = 'ADMINISTRATOR'
               THEN CMS.lastAuthor
               ELSE CMS.created
          END
      )) = UPPER(TRIM(UP.UserName))
    LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.cluster_Owner_analysis AS co
      ON CAST(CMS.Document_Id AS STRING) = co.Doc_ID
    WHERE UPPER(TRIM(cms.Document_name)) NOT IN ('DOCUMENT NOT FOUND ON SERVER')
      AND UP.BU IN ('COMCE-Germany,Austria, Switzerl,CEE')
      AND CAST(CMS.Document_Id AS STRING) IN (
        SELECT DISTINCT CAST(UA.WEBI_Doc_ID AS STRING)
        FROM dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit AS UA
        INNER JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS UP2
          ON UPPER(TRIM(UA.UserName)) = UPPER(TRIM(UP2.UserName))
        WHERE UP2.BU IN ('COMCE-Germany,Austria, Switzerl,CEE')
      )
)

SELECT 'Accessed additionaly' AS category, COALESCE(CAST(CAST(cluster AS INT) AS STRING), 'UNCLUSTERED') AS cluster, COUNT(DISTINCT Doc_id) AS reports
FROM dataplatform01_central_dev_catalog_01.custom_sap_bo.cluster_analysis 
WHERE Doc_id NOT IN (SELECT Doc_ID FROM owned)
  AND BU = UPPER('COMCE-Germany,Austria, Switzerl,CEE')
GROUP BY cluster

UNION ALL

SELECT 'Owned' AS category, COALESCE(CAST(CAST(cluster AS INT) AS STRING), 'UNCLUSTERED') AS cluster, COUNT(DISTINCT Doc_ID) AS reports
FROM owned
GROUP BY cluster
ORDER BY category ASC, cluster ASC

In [0]:
import matplotlib.pyplot as plt
import pandas as pd

# --- Get distinct BUs from Owned Reports ---

pdf_bus = spark.sql(f"""
  SELECT DISTINCT UP.BU
  FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata AS CMS
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit AS UA
    ON UPPER(TRIM(CMS.Document_Id)) = UPPER(TRIM(CAST(UA.WEBI_Doc_ID AS STRING)))
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS UP
    ON UPPER(TRIM(UP.UserName)) = UPPER(TRIM(UA.UserName))
  WHERE UPPER(TRIM(cms.Document_name)) NOT IN ('DOCUMENT NOT FOUND ON SERVER')
    AND UPPER(UP.BU) IN ('{bu_filter_sql}')
  ORDER BY BU
""").toPandas()
bu_list = pdf_bus['BU'].dropna().tolist()

# --- Query 1: Document access vs ownership ---
pdf = spark.sql(f"""
  WITH Owned_Reports AS (
    SELECT DISTINCT
      cms.Document_Id,
      CASE WHEN S1.document_id IS NULL THEN 'N' ELSE 'Y' END AS scheduled
    FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata AS CMS
    LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS UP
      ON UPPER(TRIM(
          CASE WHEN UPPER(TRIM(CMS.created)) = 'ADMINISTRATOR'
               THEN CMS.lastAuthor
               ELSE CMS.created
          END
      )) = UPPER(TRIM(UP.UserName))
    LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_extracted AS S1
      ON UPPER(TRIM(CMS.Document_Id)) = UPPER(TRIM(S1.document_id))
    WHERE UPPER(TRIM(cms.Document_name)) NOT IN ('DOCUMENT NOT FOUND ON SERVER')
      AND UPPER(UP.BU) IN ('{bu_filter_sql}')
  )

  SELECT
    'Only Access' AS category,
    COUNT(DISTINCT CMS.Document_Id) AS Document_cnt
  FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata AS CMS
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit AS UA
    ON UPPER(TRIM(CMS.Document_Id)) = UPPER(TRIM(CAST(UA.WEBI_Doc_ID AS STRING)))
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS UP
    ON UPPER(TRIM(UP.UserName)) = UPPER(TRIM(UA.UserName))
  WHERE UPPER(TRIM(cms.Document_name)) NOT IN ('DOCUMENT NOT FOUND ON SERVER')
    AND UPPER(UP.BU) IN ('{bu_filter_sql}')
    AND cms.Document_Id NOT IN (SELECT DISTINCT Document_Id FROM Owned_Reports)

  UNION ALL

  SELECT
    'Access Owned' AS category,
    COUNT(DISTINCT CMS.Document_Id) AS Document_cnt
  FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata AS CMS
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit AS UA
    ON UPPER(TRIM(CMS.Document_Id)) = UPPER(TRIM(CAST(UA.WEBI_Doc_ID AS STRING)))
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS UP
    ON UPPER(TRIM(UP.UserName)) = UPPER(TRIM(UA.UserName))
  WHERE UPPER(TRIM(cms.Document_name)) NOT IN ('DOCUMENT NOT FOUND ON SERVER')
    AND UPPER(UP.BU) IN ('{bu_filter_sql}')
    AND cms.Document_Id IN (SELECT DISTINCT Document_Id FROM Owned_Reports)
""").toPandas()

# --- Query 2: Owned reports - normal vs scheduled ---
pdf2 = spark.sql(f"""
  WITH Owned_Reports AS (
    SELECT DISTINCT
      cms.Document_Id,
      CASE WHEN S1.document_id IS NULL THEN 'N' ELSE 'Y' END AS scheduled,
      S1.Active_Schedule as schedule_active
    FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata AS CMS
    LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS UP
      ON UPPER(TRIM(
          CASE WHEN UPPER(TRIM(CMS.created)) = 'ADMINISTRATOR'
               THEN CMS.lastAuthor
               ELSE CMS.created
          END
      )) = UPPER(TRIM(UP.UserName))
    LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_extracted AS S1
      ON UPPER(TRIM(CMS.Document_Id)) = UPPER(TRIM(S1.document_id))
    WHERE UPPER(TRIM(cms.Document_name)) NOT IN ('DOCUMENT NOT FOUND ON SERVER')
      AND UPPER(UP.BU) IN ('{bu_filter_sql}') 
  )

  SELECT
    'Total owned normal reports' AS category,
    COUNT(DISTINCT Document_Id) AS Document_cnt
  FROM Owned_Reports
  WHERE scheduled = 'N'
  or( scheduled ='Y' and schedule_active = false)

  UNION ALL

  SELECT
    'Total owned scheduled reports' AS category,
    COUNT(DISTINCT Document_Id) AS Document_cnt
  FROM Owned_Reports
  WHERE scheduled = 'Y'
  AND  schedule_active = true
""").toPandas()

# --- Combined Chart with BU table side by side ---
access_owned = pdf.loc[pdf['category'] == 'Access Owned', 'Document_cnt'].values[0]
only_access = pdf.loc[pdf['category'] == 'Only Access', 'Document_cnt'].values[0]
normal = pdf2.loc[pdf2['category'] == 'Total owned normal reports', 'Document_cnt'].values[0]
scheduled = pdf2.loc[pdf2['category'] == 'Total owned scheduled reports', 'Document_cnt'].values[0]

fig, (ax, ax_table) = plt.subplots(1, 2, figsize=(10, 7), gridspec_kw={'width_ratios': [2, 1.2]})

width = 0.25
positions = [0, 0.5]

# Bar 1: Access distribution
ax.bar(positions[0], access_owned, width=width, color='#336699', label='Reports Owned')
ax.bar(positions[0], only_access, width=width, bottom=access_owned, color='#FABE6E', label='Reports Outside BU')
ax.text(positions[0], access_owned / 2, f'{access_owned:,}', ha='center', va='center', fontweight='bold', fontsize=12, color='white')
ax.text(positions[0], access_owned + only_access / 2, f'{only_access:,}', ha='center', va='center', fontweight='bold', fontsize=12, color='black')

# Bar 2: Owned reports breakdown
ax.bar(positions[1], normal, width=width, color='#4D7DAF', label='Owned Reports')
ax.bar(positions[1], scheduled, width=width, bottom=normal, color='#FFD700', label='Owned Reports with Schedule')
ax.text(positions[1], normal / 2, f'{normal:,}', ha='center', va='center', fontweight='bold', fontsize=12, color='white')
ax.text(positions[1], normal + scheduled / 2, f'{scheduled:,}', ha='center', va='center', fontweight='bold', fontsize=12, color='black')

ax.set_xticks(positions)
ax.set_xticklabels(['BO AUDIT Log\nAccessd Reports', 'BO Reports INVENTORY\nWith Schedules'])
ax.set_ylabel('Reports Count')
ax.set_title(f'{BU_Group} - Overview')
ax.legend(title='Category', fontsize=8, title_fontsize=8, loc='upper left')

# --- BU Included Table (from SQL result) ---
ax_table.axis('off')
ax_table.set_title('BU Included', fontsize=8, fontweight='bold', loc='left')

table_data = [[bu] for bu in bu_list]
tbl = ax_table.table(
    cellText=table_data,
    colLabels=['Business Unit'],
    loc='upper left',
    cellLoc='left',
    colWidths=[1.0]
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(6)
tbl.scale(1, 1.3)

# Style header
for (row, col), cell in tbl.get_celld().items():
    if row == 0:
        cell.set_facecolor('#336699')
        cell.set_text_props(color='white', fontweight='bold')
    else:
        cell.set_facecolor('#F5F8FC')
    cell.set_edgecolor('#CCCCCC')

plt.tight_layout()
plt.show()

In [0]:
%sql
 WITH owned AS (
    SELECT DISTINCT co.cluster, CAST(cms.Document_Id AS STRING) AS Doc_ID
    FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata AS CMS
    LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS UP
      ON UPPER(TRIM(
          CASE WHEN UPPER(TRIM(CMS.created)) = 'ADMINISTRATOR'
               THEN CMS.lastAuthor
               ELSE CMS.created
          END
      )) = UPPER(TRIM(UP.UserName))
    LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.cluster_Owner_analysis AS co
      ON CAST(CMS.Document_Id AS STRING) = co.Doc_ID
    WHERE UPPER(TRIM(cms.Document_name)) NOT IN ('DOCUMENT NOT FOUND ON SERVER')
      AND UPPER(UP.BU) IN (SELECT UPPER(bu) FROM vw_bu_filter)
      AND CAST(CMS.Document_Id AS STRING) IN (
        SELECT DISTINCT CAST(UA.WEBI_Doc_ID AS STRING)
        FROM dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit AS UA
        INNER JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS UP2
          ON UPPER(TRIM(UA.UserName)) = UPPER(TRIM(UP2.UserName))
        WHERE UPPER(TRIM(UP2.BU)) IN (SELECT UPPER(bu) FROM vw_bu_filter)
      )
  )

  SELECT 'Accessed additionaly' AS category, COALESCE(CAST(CAST(cluster AS INT) AS STRING), 'UNCLUSTERED') AS cluster, COUNT(DISTINCT Doc_id) AS reports
  FROM dataplatform01_central_dev_catalog_01.custom_sap_bo.cluster_analysis
  WHERE Doc_id NOT IN (SELECT Doc_ID FROM owned)
    AND BU in (SELECT UPPER(bu) FROM vw_bu_filter)
  GROUP BY cluster

  UNION ALL

  SELECT 'Owned' AS category, COALESCE(CAST(CAST(cluster AS INT) AS STRING), 'UNCLUSTERED') AS cluster, COUNT(DISTINCT Doc_ID) AS reports
  FROM owned
  GROUP BY cluster
  ORDER BY category ASC, cluster ASC

In [0]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

pdf_bus = spark.sql(f"""
  SELECT DISTINCT UP.BU
  FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata AS CMS
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit AS UA
    ON UPPER(TRIM(CMS.Document_Id)) = UPPER(TRIM(CAST(UA.WEBI_Doc_ID AS STRING)))
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS UP
    ON UPPER(TRIM(UP.UserName)) = UPPER(TRIM(UA.UserName))
  WHERE UPPER(TRIM(cms.Document_name)) NOT IN ('DOCUMENT NOT FOUND ON SERVER')
    AND UPPER(UP.BU) IN ('{bu_filter_sql}')
  ORDER BY BU
""").toPandas()
bu_list = pdf_bus['BU'].dropna().tolist()

# --- Query: Accessed additionally vs Owned by cluster (aligned with Cell 3) ---
pdf3 = spark.sql(f"""
  WITH owned AS (
    SELECT DISTINCT co.cluster, CAST(cms.Document_Id AS STRING) AS Doc_ID
    FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata AS CMS
    LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS UP
      ON UPPER(TRIM(
          CASE WHEN UPPER(TRIM(CMS.created)) = 'ADMINISTRATOR'
               THEN CMS.lastAuthor
               ELSE CMS.created
          END
      )) = UPPER(TRIM(UP.UserName))
    LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.cluster_Owner_analysis AS co
      ON CAST(CMS.Document_Id AS STRING) = co.Doc_ID
    WHERE UPPER(TRIM(cms.Document_name)) NOT IN ('DOCUMENT NOT FOUND ON SERVER')
      AND UPPER(UP.BU) IN ('{bu_filter_sql}')
      AND CAST(CMS.Document_Id AS STRING) IN (
        SELECT DISTINCT CAST(UA.WEBI_Doc_ID AS STRING)
        FROM dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit AS UA
        INNER JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS UP2
          ON UPPER(TRIM(UA.UserName)) = UPPER(TRIM(UP2.UserName))
        WHERE UPPER(TRIM(UP2.BU)) IN ('{bu_filter_sql}')
      )
  )

  SELECT 'Accessed additionaly' AS category, COALESCE(CAST(CAST(cluster AS INT) AS STRING), 'UNCLUSTERED') AS cluster, COUNT(DISTINCT Doc_id) AS reports
  FROM dataplatform01_central_dev_catalog_01.custom_sap_bo.cluster_analysis
  WHERE Doc_id NOT IN (SELECT Doc_ID FROM owned)
    AND UPPER(BU)  in ('{bu_filter_sql}')
  GROUP BY cluster

  UNION ALL

  SELECT 'Owned' AS category, COALESCE(CAST(CAST(cluster AS INT) AS STRING), 'UNCLUSTERED') AS cluster, COUNT(DISTINCT Doc_ID) AS reports
  FROM owned
  GROUP BY cluster
  ORDER BY category ASC, cluster ASC
""").toPandas()

# --- Stacked Bar Chart: Accessed additionally on top of Owned ---
pivot = pdf3.pivot(index='cluster', columns='category', values='reports').fillna(0)

# Sort: numeric clusters first, then UNCLUSTERED at the end
def sort_key(x):
    try:
        return (0, int(x))
    except ValueError:
        return (1, 0)

pivot = pivot.loc[sorted(pivot.index, key=sort_key)]
clusters = pivot.index.tolist()
x = np.arange(len(clusters)) * 0.7  # Tighter spacing between bars

owned_vals = pivot.get('Owned', pd.Series([0]*len(clusters))).values
accessed_vals = pivot.get('Accessed additionaly', pd.Series([0]*len(clusters))).values

fig, (ax, ax_table) = plt.subplots(1, 2, figsize=(12, 6), gridspec_kw={'width_ratios': [2.5, 1.5]})

width = 0.45
bars_owned = ax.bar(x, owned_vals, width=width, color='#336699', label='Owned')
bars_accessed = ax.bar(x, accessed_vals, width=width, bottom=owned_vals, color='#FABE6E', label='Accessed additionaly')

# Proportional offset based on y-axis range
y_max = max(owned_vals + accessed_vals)
label_offset = y_max * 0.02  # 2% of chart height

# Add value labels - place above bar if segment is too small
min_height_for_inside = y_max * 0.08  # 8% of chart height as threshold
for i, (o, a) in enumerate(zip(owned_vals, accessed_vals)):
    total_height = o + a
    if o > 0:
        if o < min_height_for_inside:
            # Both segments too small - show side by side above bar with | separator
            offset = total_height + label_offset
            if a > 0:
                ax.text(x[i] - 0.1, offset, f'{int(o):,}', ha='center', va='bottom', fontsize=8, fontweight='bold', color='#336699')
                ax.text(x[i], offset, '|', ha='center', va='bottom', fontsize=8, fontweight='bold', color='#336699')
                ax.text(x[i] + 0.1, offset, f'{int(a):,}', ha='center', va='bottom', fontsize=8, fontweight='bold', color='#E8960C')
            else:
                ax.text(x[i], offset, f'{int(o):,}', ha='center', va='bottom', fontsize=8, fontweight='bold', color='#336699')
        else:
            ax.text(x[i], o / 2, f'{int(o):,}', ha='center', va='center', fontsize=9, fontweight='bold', color='white')
            if a > 0:
                if a < min_height_for_inside:
                    ax.text(x[i], total_height + label_offset, f'{int(a):,}', ha='center', va='bottom', fontsize=8, fontweight='bold', color='#E8960C')
                else:
                    ax.text(x[i], o + a / 2, f'{int(a):,}', ha='center', va='center', fontsize=9, fontweight='bold', color='black')
    elif a > 0:
        if a < min_height_for_inside:
            ax.text(x[i], a + label_offset, f'{int(a):,}', ha='center', va='bottom', fontsize=8, fontweight='bold', color='#E8960C')
        else:
            ax.text(x[i], a / 2, f'{int(a):,}', ha='center', va='center', fontsize=9, fontweight='bold', color='black')

ax.set_xlabel('Cluster')
ax.set_ylabel('Reports Count')
ax.set_title(f'{BU_Group} - Reports by Cluster')
ax.set_xticks(x)
ax.set_xticklabels(clusters)
ax.legend(title='Category', fontsize=10, loc='upper left')

# --- BU Included Table ---
ax_table.axis('off')
ax_table.set_title('BU Included', fontsize=8, fontweight='bold', loc='left')

table_data = [[bu] for bu in bu_list]
tbl = ax_table.table(
    cellText=table_data,
    colLabels=['Business Unit'],
    loc='upper left',
    cellLoc='left',
    colWidths=[0.6]
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(6)
tbl.scale(1, 1.3)

# Style header
for (row, col), cell in tbl.get_celld().items():
    if row == 0:
        cell.set_facecolor('#336699')
        cell.set_text_props(color='white', fontweight='bold')
    else:
        cell.set_facecolor('#F5F8FC')
    cell.set_edgecolor('#CCCCCC')

plt.tight_layout()
plt.show()

In [0]:
%sql
    
WITH eventCategorized AS (
  SELECT
    ua1.*,
    COALESCE(CAST(CAST(co.cluster AS INT) AS STRING), 'UNCLUSTERED') AS Report_Cluster,
    CASE
      WHEN event_type IN ('Edit', 'Deliver', 'Modify', 'Save', 'Delete', 'Create') THEN 'Editor_events'
      ELSE 'Viewer_events'
    END AS usage_category
  FROM dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit AS ua1
  LEFT JOIN dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata AS CMS
    ON ua1.WEBI_Doc_ID = CAST(CMS.Document_Id AS STRING)
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.cluster_Owner_analysis AS co
    ON CAST(ua1.WEBI_Doc_ID AS STRING) = co.Doc_ID
  WHERE CMS.Document_name NOT IN ('DOCUMENT NOT FOUND ON SERVER')
    AND CMS.Document_Id IS NOT NULL
)

SELECT
  ec.UserName as User_ID, up1.DisplayName as User_Name, up1.BU as User_Bu, up1.Title as User_Role,
  SUM(CASE WHEN usage_category = 'Viewer_events' THEN 1 ELSE 0 END) AS Viewer_events,
  COUNT(DISTINCT CASE WHEN usage_category = 'Viewer_events' THEN ec.WEBI_Doc_ID END) AS Viewer_report_cnt
FROM eventCategorized AS ec
LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS up1
  ON ec.UserName = up1.UserName
WHERE Upper(up1.BU) in (Select upper(BU) from vw_bu_filter)
and up1.UserName not in ('DESWAH1','DEJSCH6','DECGAS2','DEMJAN2','CZLBEC1','DEIRAT2','CHPSTU1')
GROUP BY ec.UserName, up1.DisplayName, up1.BU, up1.Title
ORDER BY Viewer_report_cnt DESC
limit 7


In [0]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from matplotlib.colors import LinearSegmentedColormap

# --- Query: Top Editors with cluster detail ---
pdf_editors = spark.sql(f"""
  WITH eventCategorized AS (
    SELECT
      ua1.*,
      COALESCE(CAST(CAST(co.cluster AS INT) AS STRING), 'UNCLUSTERED') AS Report_Cluster,
      CASE
        WHEN event_type IN ('Edit', 'Deliver', 'Modify', 'Save', 'Delete', 'Create') THEN 'Editor_events'
        ELSE 'Viewer_events'
      END AS usage_category
    FROM dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit AS ua1
    LEFT JOIN dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata AS CMS
      ON ua1.WEBI_Doc_ID = CAST(CMS.Document_Id AS STRING)
    LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.cluster_Owner_analysis AS co
      ON CAST(ua1.WEBI_Doc_ID AS STRING) = co.Doc_ID
    WHERE CMS.Document_name NOT IN ('DOCUMENT NOT FOUND ON SERVER')
      AND CMS.Document_Id IS NOT NULL
  )

  SELECT
    ec.UserName AS User_ID,
    up1.DisplayName AS User_Name,
    up1.BU AS User_BU,
    up1.Title AS User_Role,
    ec.Report_Cluster,
    COUNT(DISTINCT CASE WHEN usage_category = 'Editor_events' THEN WEBI_Doc_ID END) AS Editor_report_cnt,
    sum(ec.Events) as Interactions_cnt
  FROM eventCategorized AS ec
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS up1
    ON ec.UserName = up1.UserName
  WHERE upper(up1.BU) IN ('{bu_filter_sql}')
  GROUP BY ec.UserName, up1.DisplayName, up1.BU, up1.Title, ec.Report_Cluster
  HAVING COUNT(DISTINCT CASE WHEN usage_category = 'Editor_events' THEN WEBI_Doc_ID END) > 0
""").toPandas()

# --- Identify top 7 editors by total report count ---
editor_totals = pdf_editors.groupby(['User_ID', 'User_Name', 'User_Role']).agg(
    Editor_report_cnt=('Editor_report_cnt', 'sum'),
    Interactions_cnt=('Interactions_cnt', 'sum')
).reset_index()
editor_totals = editor_totals.sort_values('Editor_report_cnt', ascending=False).head(7)
top_editor_ids = editor_totals['User_ID'].tolist()

pdf_editors_top = pdf_editors[pdf_editors['User_ID'].isin(top_editor_ids)].copy()

# --- Query: Top Viewers (excluding editors) with cluster detail ---
editor_ids_sql = ",".join([f"'{uid}'" for uid in top_editor_ids])
pdf_viewers = spark.sql(f"""
  WITH eventCategorized AS (
    SELECT
      ua1.*,
      COALESCE(CAST(CAST(co.cluster AS INT) AS STRING), 'UNCLUSTERED') AS Report_Cluster,
      CASE
        WHEN event_type IN ('Edit', 'Deliver', 'Modify', 'Save', 'Delete', 'Create') THEN 'Editor_events'
        ELSE 'Viewer_events'
      END AS usage_category
    FROM dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit AS ua1
    LEFT JOIN dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata AS CMS
      ON ua1.WEBI_Doc_ID = CAST(CMS.Document_Id AS STRING)
    LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.cluster_Owner_analysis AS co
      ON CAST(ua1.WEBI_Doc_ID AS STRING) = co.Doc_ID
    WHERE CMS.Document_name NOT IN ('DOCUMENT NOT FOUND ON SERVER')
      AND CMS.Document_Id IS NOT NULL
  )

  SELECT
    ec.UserName AS User_ID,
    up1.DisplayName AS User_Name,
    up1.BU AS User_BU,
    up1.Title AS User_Role,
    ec.Report_Cluster,
    COUNT(DISTINCT CASE WHEN usage_category = 'Viewer_events' THEN WEBI_Doc_ID END) AS Viewer_report_cnt,
    sum(ec.Events) as Interactions_cnt
  FROM eventCategorized AS ec
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS up1
    ON ec.UserName = up1.UserName
  WHERE upper(up1.BU) in ( '{bu_filter_sql}')
    AND up1.UserName NOT IN ({editor_ids_sql})
  GROUP BY ec.UserName, up1.DisplayName, up1.BU, up1.Title, ec.Report_Cluster
  HAVING COUNT(DISTINCT CASE WHEN usage_category = 'Viewer_events' THEN WEBI_Doc_ID END) > 0
""").toPandas()

# --- Identify top 7 viewers by total report count ---
viewer_totals = pdf_viewers.groupby(['User_ID', 'User_Name', 'User_Role']).agg(
    Viewer_report_cnt=('Viewer_report_cnt', 'sum'),
    Interactions_cnt=('Interactions_cnt', 'sum')
).reset_index()
viewer_totals = viewer_totals.sort_values('Viewer_report_cnt', ascending=False).head(7)
top_viewer_ids = viewer_totals['User_ID'].tolist()

pdf_viewers_top = pdf_viewers[pdf_viewers['User_ID'].isin(top_viewer_ids)].copy()

# --- Smooth gradient palettes using colormaps (0-11 + UNCLUSTERED) ---
def generate_palette(cmap_name, n=12):
    """Generate a smooth gradient palette from a matplotlib colormap."""
    cmap = plt.cm.get_cmap(cmap_name, n + 2)
    palette = {}
    for i in range(n):
        rgba = cmap(0.15 + (i / (n - 1)) * 0.77)
        palette[str(i)] = '#{:02x}{:02x}{:02x}'.format(int(rgba[0]*255), int(rgba[1]*255), int(rgba[2]*255))
    palette['UNCLUSTERED'] = '#D4D4D4'
    return palette

def get_text_color(hex_color):
    """Return white for dark backgrounds, black for light backgrounds based on luminance."""
    hex_color = hex_color.lstrip('#')
    r, g, b = int(hex_color[0:2], 16), int(hex_color[2:4], 16), int(hex_color[4:6], 16)
    luminance = (0.299 * r + 0.587 * g + 0.114 * b) / 255
    return 'white' if luminance < 0.5 else 'black'

blue_palette = generate_palette('Blues', 12)
orange_palette = generate_palette('Oranges', 12)

# Sort clusters: numeric first, UNCLUSTERED last
def cluster_sort_key(c):
    try:
        return (0, int(c))
    except ValueError:
        return (1, 0)

# --- Helper function: stacked horizontal bar chart by cluster ---
def plot_cluster_barh(pdf, user_col, count_col, title, top_ids, totals_df, palette):
    user_order = totals_df.sort_values(count_col, ascending=True)['User_ID'].tolist()
    user_names = totals_df.set_index('User_ID')['User_Name'].to_dict()
    
    pivot = pdf.pivot_table(index='User_ID', columns='Report_Cluster', values=count_col, aggfunc='sum', fill_value=0)
    pivot = pivot.reindex(user_order)
    
    all_clusters = sorted(pivot.columns.tolist(), key=cluster_sort_key)
    pivot = pivot[all_clusters]
    
    # For each user, find their top 2 clusters (excl UNCLUSTERED)
    numeric_clusters = [c for c in all_clusters if c != 'UNCLUSTERED']
    user_top2 = {}
    for uid in user_order:
        user_vals = pivot.loc[uid, numeric_clusters].sort_values(ascending=False)
        user_top2[uid] = user_vals.head(2).index[user_vals.head(2) > 0].tolist()
    
    fig, ax = plt.subplots(figsize=(10, 4))
    y_pos = np.arange(len(user_order))
    bar_height = 0.5
    
    left = np.zeros(len(user_order))
    bar_positions = {}
    for cluster in all_clusters:
        vals = pivot[cluster].values
        color = palette.get(cluster, '#D4D4D4')
        ax.barh(y_pos, vals, left=left, height=bar_height, color=color, edgecolor='white', linewidth=0.5, label=cluster)
        bar_positions[cluster] = (left.copy(), vals, color)
        left += vals
    
    # Add in-bar labels for each user's top 2 clusters with contrast-aware color
    for i, uid in enumerate(user_order):
        for cluster in user_top2.get(uid, []):
            lefts, vals, color = bar_positions[cluster]
            v = vals[i]
            if v > 0:
                txt_color = get_text_color(color)
                ax.text(lefts[i] + v / 2, y_pos[i], f'{int(v)}', ha='center', va='center',
                        fontsize=7, fontweight='bold', color=txt_color)
    
    # Total labels at end of bar - proportional offset
    x_max = max(left) if max(left) > 0 else 1
    label_gap = x_max * 0.02  # 2% of max bar length
    for i, uid in enumerate(user_order):
        total = int(left[i])
        ax.text(left[i] + label_gap, y_pos[i], f'{total:,}', va='center', fontsize=9, fontweight='bold', color='#333333')
    
    ax.set_yticks(y_pos)
    ax.set_yticklabels([user_names.get(uid, uid) or uid for uid in user_order], fontsize=10)
    ax.set_xlabel('Distinct Reports')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.legend(title='Cluster', fontsize=6, loc='lower right', ncol=6, framealpha=0.9, columnspacing=0.8, handlelength=1.0)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    plt.show()

# --- Figure 1: Top Editors by cluster (blue gradient) ---
plot_cluster_barh(pdf_editors_top, 'User_ID', 'Editor_report_cnt',
                  f'{BU_Group} - Top 7 Editors by Report Count (by Cluster)',
                  top_editor_ids, editor_totals, blue_palette)

# --- Figure 2: Top Viewers by cluster (orange gradient) ---
plot_cluster_barh(pdf_viewers_top, 'User_ID', 'Viewer_report_cnt',
                  f'{BU_Group} - Top 7 Viewers by Report Count (by Cluster, excl. Editors)',
                  top_viewer_ids, viewer_totals, orange_palette)

# --- Display summary table ---
editor_totals_disp = editor_totals.rename(columns={'User_Name': 'Name', 'User_Role': 'Role', 'Interactions_cnt': 'Interactions', 'Editor_report_cnt': 'Reports'})
editor_totals_disp.insert(0, 'Category', 'Editor')
editor_totals_disp = editor_totals_disp.drop(columns=['User_ID'])

viewer_totals_disp = viewer_totals.rename(columns={'User_Name': 'Name', 'User_Role': 'Role', 'Interactions_cnt': 'Interactions', 'Viewer_report_cnt': 'Reports'})
viewer_totals_disp.insert(0, 'Category', 'Viewer')
viewer_totals_disp = viewer_totals_disp.drop(columns=['User_ID'])

pdf_combined = pd.concat([editor_totals_disp, viewer_totals_disp], ignore_index=True)

# Move Reports to last column
cols = [c for c in pdf_combined.columns if c != 'Reports'] + ['Reports']
pdf_combined = pdf_combined[cols]

# Format numeric columns with thousand separators
for col in pdf_combined.select_dtypes(include='number').columns:
    pdf_combined[col] = pdf_combined[col].apply(lambda x: f'{int(x):,}')

display(pdf_combined)

In [0]:
%sql
WITH labled_schedule AS (
  SELECT
    sd1.sent_to,
    up1.EmailAddress,
    CASE
      WHEN sd1.destination_type = 'mail' AND UPPER(sd1.sent_to) = UPPER(up1.EmailAddress) THEN 'Email self'
      WHEN sd1.destination_type = 'mail' THEN 'Email others'
      WHEN sd1.destination_type = 'filesystem' THEN 'Shared Location'
      ELSE NULL
    END AS Consumption,
    sd1.*
  FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_schedule_details_enriched AS sd1
  left join dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_extracted as se1
  ON sd1.document_id = se1.document_id
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS up1
    ON sd1.document_created_by = up1.UserName
  WHERE upper(up1.BU) in (select upper(bu) from vw_bu_filter)
  and se1.Active_Schedule= TRUE
)

SELECT
  CASE
    WHEN sd2.email_delivery = TRUE AND labled_schedule.Consumption IS NOT NULL THEN labled_schedule.Consumption
    WHEN sd2.BO_delivery=TRUE THEN 'SAP BO'
    WHEN sd2.file_delivery = TRUE THEN 'Shared Drive'
    WHEN labled_schedule.Consumption IS NULL THEN 'Paused'
    ELSE NULL
  END AS Category,
  COUNT(DISTINCT CMS.Document_Id) AS Reports_cnt,
  COUNT(DISTINCT CMS.created) AS Unique_Owner_cnt,
  ARRAY_JOIN(SORT_ARRAY(COLLECT_SET(UP.DisplayName)), ' | ') AS Report_Owners
FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata AS CMS
LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS UP
  ON UPPER(TRIM(
          CASE WHEN UPPER(TRIM(CMS.created)) = 'ADMINISTRATOR'
               THEN CMS.lastAuthor
               ELSE CMS.created
          END
      )) = UPPER(TRIM(UP.UserName))
LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_extracted AS sd2
  ON CAST(cms.Document_Id AS STRING) = sd2.document_id
LEFT JOIN labled_schedule
  ON sd2.document_id = labled_schedule.document_id
WHERE upper(UP.BU) in (select upper(bu) from vw_bu_filter)
  AND sd2.document_id IS NOT NULL
  AND sd2.Active_Schedule= TRUE
GROUP BY
  CASE
    WHEN sd2.email_delivery = TRUE AND labled_schedule.Consumption IS NOT NULL THEN labled_schedule.Consumption
    WHEN sd2.BO_delivery=TRUE THEN 'SAP BO'
    WHEN sd2.file_delivery = TRUE THEN 'Shared Drive'
    WHEN labled_schedule.Consumption IS NULL THEN 'Paused'
    ELSE NULL
  END
ORDER BY
  CASE
    WHEN
      Category='SAP BO' THEN 1
    WHEN
      Category='Email self' THEN 2
    WHEN
      Category='Email others' THEN 3
    WHEN
     Category='Shared Drive' THEN 4
    WHEN
      Category='Paused' THEN 5
    ELSE 6
  END

In [0]:
%sql
WITH labled_schedule AS (
  SELECT
    up1.BU,
    sd1.sent_to,
    up1.EmailAddress,
    CASE
      WHEN sd1.destination_type = 'mail' AND UPPER(sd1.sent_to) = UPPER(up1.EmailAddress) THEN 'Email self'
      WHEN sd1.destination_type = 'mail' THEN 'Email others'
      WHEN sd1.destination_type = 'filesystem' THEN 'Shared Drive'
      ELSE NULL
    END AS Consumption,
    sd1.*
  FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_schedule_details_enriched AS sd1
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_extracted AS sd2
  ON CAST(sd1.Document_Id AS STRING) = sd2.document_id
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS up1
    ON sd1.document_created_by = up1.UserName
  WHERE sd2.Active_Schedule = TRUE
),

base AS (
  SELECT
    UP.BU AS BusinessUnit,
    UP.DisplayName AS User_name,
    CASE
      WHEN sd2.destination_type = 'mail' AND labled_schedule.Consumption IS NOT NULL THEN labled_schedule.Consumption
      WHEN sd2.destination_type = 'inbox' OR sd2.destination_type IS NULL THEN 'SAP BO'
      WHEN sd2.destination_type = 'filesystem' THEN 'Shared Drive'
      WHEN labled_schedule.Consumption IS NULL THEN 'Paused'
      ELSE sd2.destination_type
    END AS Category,
    COUNT(DISTINCT CMS.Document_Id) AS Reports_cnt
  FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata AS CMS
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS UP
    ON UPPER(TRIM(CMS.created)) = UPPER(TRIM(UP.UserName))
  LEFT JOIN dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_schedule_details AS sd2
    ON CAST(cms.Document_Id AS STRING) = sd2.document_id
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_extracted AS sd3
    ON CAST(sd3.Document_Id AS STRING) = sd2.document_id
  LEFT JOIN labled_schedule
    ON sd2.document_id = labled_schedule.document_id
  WHERE sd2.document_id IS NOT NULL
    AND sd3.Active_Schedule = TRUE
  GROUP BY
    UP.BU,
    UP.DisplayName,
    CASE
      WHEN sd2.destination_type = 'mail' AND labled_schedule.Consumption IS NOT NULL THEN labled_schedule.Consumption
      WHEN sd2.destination_type = 'inbox' OR sd2.destination_type IS NULL THEN 'SAP BO'
      WHEN sd2.destination_type = 'filesystem' THEN 'Shared Drive'
      WHEN labled_schedule.Consumption IS NULL THEN 'Paused'
      ELSE sd2.destination_type
    END
),

base_with_total AS (
  SELECT * FROM base
  UNION ALL
  SELECT
    UP.BU AS BusinessUnit,
    UP.DisplayName AS User_name,
    'Total' AS Category,
    COUNT(DISTINCT CMS.Document_Id) AS Reports_cnt
  FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata AS CMS
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS UP
    ON UPPER(TRIM(CMS.created)) = UPPER(TRIM(UP.UserName))
  LEFT JOIN dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_schedule_details AS sd2
    ON CAST(cms.Document_Id AS STRING) = sd2.document_id
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_extracted AS sd3
    ON sd3.document_id = sd2.document_id
  LEFT JOIN labled_schedule
    ON sd2.document_id = labled_schedule.document_id
  WHERE sd2.document_id IS NOT NULL
    AND sd3.Active_Schedule = TRUE
  GROUP BY UP.BU, UP.DisplayName
)

SELECT *
FROM base_with_total
PIVOT (
  SUM(Reports_cnt)
  FOR Category IN (
    'SAP BO' AS SAP_BO,
    'Email self' AS Email_self,
    'Email others' AS Email_others,
    'Shared Drive' AS Shared_Drive,
    'Paused' AS Paused,
    'Total' AS Total
  )
)
ORDER BY BusinessUnit, User_name

In [0]:
import matplotlib.pyplot as plt
import pandas as pd

# --- Query: Schedule Report Owners by BU (reports per category) ---
pdf_sched_owners = spark.sql(f"""
WITH labled_schedule AS (
  SELECT
    sd1.sent_to,
    up1.EmailAddress,
    CASE
      WHEN sd1.destination_type = 'mail' AND UPPER(sd1.sent_to) = UPPER(up1.EmailAddress) THEN 'Email self'
      WHEN sd1.destination_type = 'mail' THEN 'Email multiple users'
      WHEN sd1.destination_type = 'filesystem' THEN 'Shared Drive'
      ELSE NULL
    END AS Consumption,
    sd1.*
  FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_schedule_details_enriched AS sd1
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_extracted AS sd2
  ON CAST(sd1.Document_Id AS STRING) = sd2.document_id
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS up1
    ON sd1.document_created_by = up1.UserName
  WHERE upper(up1.BU) in ( '{bu_filter_sql}')
  AND sd2.Active_Schedule = TRUE
)
SELECT
  UP.BU,
  CASE
    WHEN sd2.destination_type = 'mail' AND labled_schedule.Consumption IS NOT NULL THEN labled_schedule.Consumption
    WHEN sd2.destination_type = 'inbox' OR sd2.destination_type IS NULL THEN 'SAP BO'
    WHEN sd2.destination_type = 'filesystem' THEN 'Shared Drive'
    WHEN labled_schedule.Consumption IS NULL THEN 'Paused'
    ELSE sd2.destination_type
  END AS Category,
  COUNT(DISTINCT CMS.Document_Id) AS Reports_cnt
FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata AS CMS
LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS UP
  ON UPPER(TRIM(CMS.created)) = UPPER(TRIM(UP.UserName))
LEFT JOIN dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_schedule_details AS sd2
  ON CAST(cms.Document_Id AS STRING) = sd2.document_id
LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_extracted AS sd3
  ON sd3.document_id = sd2.document_id
LEFT JOIN labled_schedule
  ON sd2.document_id = labled_schedule.document_id
WHERE upper(UP.BU) in ('{bu_filter_sql}')
  AND sd2.document_id IS NOT NULL
  AND sd3.Active_Schedule = TRUE
GROUP BY
  UP.BU,
  CASE
    WHEN sd2.destination_type = 'mail' AND labled_schedule.Consumption IS NOT NULL THEN labled_schedule.Consumption
    WHEN sd2.destination_type = 'inbox' OR sd2.destination_type IS NULL THEN 'SAP BO'
    WHEN sd2.destination_type = 'filesystem' THEN 'Shared Drive'
    WHEN labled_schedule.Consumption IS NULL THEN 'Paused'
    ELSE sd2.destination_type
  END
ORDER BY Category ASC, Reports_cnt DESC
""").toPandas()

# --- Separate query: Distinct users per BU (no category grouping = no double counting) ---
pdf_bu_users = spark.sql(f"""
  SELECT
    UP.BU,
    COUNT(DISTINCT CMS.created) AS User_cnt
  FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata AS CMS
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS UP
    ON UPPER(TRIM(CMS.created)) = UPPER(TRIM(UP.UserName))
  LEFT JOIN dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_schedule_details AS sd2
    ON CAST(cms.Document_Id AS STRING) = sd2.document_id
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_extracted AS sd3
    ON sd3.document_id = sd2.document_id
  WHERE upper(UP.BU) in ('{bu_filter_sql}')
    AND sd2.document_id IS NOT NULL
    AND sd3.Active_Schedule = TRUE
  GROUP BY UP.BU
""").toPandas()

bu_user_totals = pdf_bu_users.set_index('BU')['User_cnt'].to_dict()
print(f"Total Users: {sum(bu_user_totals.values())}")
# --- Pivot for stacked bar chart ---
pivot = pdf_sched_owners.pivot_table(index='BU', columns='Category', values='Reports_cnt', aggfunc='sum', fill_value=0)

# Sort by total reports descending
pivot['_total'] = pivot.sum(axis=1)
pivot = pivot.sort_values('_total', ascending=False)
pivot = pivot.drop(columns='_total')

owners = pivot.index
categories = pivot.columns
colors = {
    'Email self': '#5DADE2',
    'Email multiple users': '#1B4F72',
    'Shared Drive': '#F4D03F',
    'SAP BO': '#E67E22',
    'Paused': '#C0C0C0'
}

fig, ax = plt.subplots(figsize=(12, 6))
width = 0.4
bottom = pd.Series([0.0]*len(owners), index=owners)
for cat in categories:
    ax.bar(owners, pivot[cat], width=width, bottom=bottom, color=colors.get(cat, '#888888'), label=cat)
    bottom += pivot[cat]

# Add total users annotation above each bar
for i, bu in enumerate(owners):
    total_height = bottom[bu]
    users = bu_user_totals.get(bu, 0)
    ax.text(i, total_height + 1, f'{int(users)} users', ha='center', va='bottom', fontsize=7, color='#555555')

ax.set_ylabel('Scheduled Reports Count')
ax.set_title(f'{BU_Group} - Scheduled Reports by Owners BU')
ax.legend(title='Schedule Category', fontsize=10, loc='upper right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# --- Display table ---
display(pdf_sched_owners)